# 07 — Dynamic Delta Hedging

This notebook follows the hedge as a cash-and-stock process rather than treating delta as a static number. Every rebalance, financing charge, transaction cost, and terminal settlement remains visible.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


In [2]:
from options_lab.price_paths import simulate_gbm_paths
from options_lab.delta_hedging import delta_hedge_path

times,paths=simulate_gbm_paths(100.,.03,.20,1.,252,1,seed=2026)
result=delta_hedge_path(times,paths[0],100.,.03,.20,"call","short",transaction_cost_rate=.0005)
result.history[["time","spot","option_delta_per_unit","actual_stock_position","cash_account","portfolio_value","transaction_cost"]].head()

,time,spot,option_delta_per_unit,actual_stock_position,cash_account,portfolio_value,transaction_cost
0,0.000000,100.000000,0.598706,0.598706,-50.487165,-0.029935,0.029935
1,0.003968,99.009664,0.579119,0.579119,-48.554836,-0.025309,0.000970
2,0.007937,99.314150,0.584905,0.584905,-49.135513,-0.011034,0.000287
3,0.011905,96.973358,0.537412,0.537412,-44.538072,-0.053351,0.002303
4,0.015873,98.697639,0.572183,0.572183,-47.976981,-0.069533,0.001716


In [3]:
plt.figure(figsize=(8,5))
plt.plot(result.history.time,result.history.spot)
plt.axhline(100,linestyle="--",linewidth=1)
plt.xlabel("Time")
plt.ylabel("Underlying price")
plt.title("Selected Simulated Path")
plt.show()

/tmp/ipykernel_7762/1294346156.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
pd.Series({"terminal_pnl":result.terminal_pnl,
           "total_transaction_cost":result.total_transaction_cost,
           "option_premium":result.option_premium_per_unit,
           "option_payoff":result.option_payoff_per_unit,
           "rebalances_after_inception":result.number_of_rebalances})

terminal_pnl                   -0.841665
total_transaction_cost          0.297108
option_premium                  9.413403
option_payoff                  28.851568
rebalances_after_inception    251.000000
dtype: float64

**What I took from it.** Delta neutrality is temporary. Rebalancing more often can reduce replication error, but it also increases turnover and cost.